# Step 1: Load datasets. #

In [4]:
#-- Train directory import. --#
import os, sys, glob
train_dir = '../Datasets/Train/EngText/'
print("Checking directory........")
print("-" * 38)
#-- Check dir exists or not. --#
if not os.path.exists(train_dir):
    print(f"Error: The {train_dir} doesn't exist! Please check the directory again.")
else:
    print(f"The {train_dir} exists.")
    test_dir = '../Datasets/Test/EngText'
    if not os.path.exists(test_dir):
        print(f"Error: The {test_dir} does not exists! Please check again.")
    else:
        print(f"The {test_dir} exists")
        print("-" * 38)
        #-- Print files in each dir. --#
        train_csv_files = glob.glob(os.path.join(train_dir, "*.csv"))
        print(f"Checking files in {train_dir}...")
        for file in train_csv_files:
            print(f"Found file: {file}")
        print("-" * 38)
        print(f"Checking files in {test_dir}...")
        test_csv_files = glob.glob(os.path.join(test_dir, "*.csv"))
        for file in test_csv_files:
            print(f"Found file: {file}")


Checking directory........
--------------------------------------
The ../Datasets/Train/EngText/ exists.
The ../Datasets/Test/EngText exists
--------------------------------------
Checking files in ../Datasets/Train/EngText/...
Found file: ../Datasets/Train/EngText\Datasets_01.csv
Found file: ../Datasets/Train/EngText\Datasets_02.csv
Found file: ../Datasets/Train/EngText\Datasets_03.csv
Found file: ../Datasets/Train/EngText\Datasets_04.csv
Found file: ../Datasets/Train/EngText\Datasets_05.csv
Found file: ../Datasets/Train/EngText\Datasets_06.csv
Found file: ../Datasets/Train/EngText\Datasets_07.csv
Found file: ../Datasets/Train/EngText\Datasets_08.csv
Found file: ../Datasets/Train/EngText\Datasets_09.csv
Found file: ../Datasets/Train/EngText\Datasets_10.csv
Found file: ../Datasets/Train/EngText\Datasets_11.csv
Found file: ../Datasets/Train/EngText\Datasets_12.csv
Found file: ../Datasets/Train/EngText\Datasets_13.csv
Found file: ../Datasets/Train/EngText\Datasets_14.csv
Found file: ../D

# Step 2: Data Processing (Filter garbage file and combine train files into one Dataframe). #

In [5]:
import pandas as pd
#--- 1. Load and combine train files. ---#
train_dfs = []
print("#--- Loading Train Files ---#")
for file in train_csv_files:
    print(f"Loading: {os.path.basename(file)}")
    train_dfs.append(pd.read_csv(file))
train_raw_df = pd.concat(train_dfs, ignore_index=True)
print("-" * 38)
# --- 3. Check size of dataframe.  ---#
print(f"Train Raw Shape : {train_raw_df.shape}")

#--- Loading Train Files ---#
Loading: Datasets_01.csv
Loading: Datasets_02.csv
Loading: Datasets_03.csv
Loading: Datasets_04.csv
Loading: Datasets_05.csv
Loading: Datasets_06.csv
Loading: Datasets_07.csv
Loading: Datasets_08.csv
Loading: Datasets_09.csv
Loading: Datasets_10.csv
Loading: Datasets_11.csv
Loading: Datasets_12.csv
Loading: Datasets_13.csv
Loading: Datasets_14.csv
Loading: Datasets_15.csv
--------------------------------------
Train Raw Shape : (297960, 785)


In [7]:
import numpy as np
raw_labels = train_raw_df.iloc[:, 0].values
unique_labels = np.unique(raw_labels)
unique_labels.sort()

print(f"Unique label indices found in Train CSV : {unique_labels}")
print(f"Total unique label count                 : {len(unique_labels)}")
print(f"Min label index: {unique_labels.min()} | Max label index: {unique_labels.max()}")
print("=" * 60)

# 2. Map CSV numeric labels to actual letters (0 -> 'A', 1 -> 'B', ...)
LABEL_TO_CHAR = {int(label): chr(65 + int(label)) for label in unique_labels}

# 3. Print verification table showing each label and its frequency in Train CSV
print("=== DIRECT INSPECTION RESULTS FROM TRAIN CSV ===")
for label in unique_labels:
    char = LABEL_TO_CHAR[int(label)]
    count = (raw_labels == label).sum()
    print(f"CSV Label: {int(label):2d}  -->  Letter: '{char}'  |  Sample Count: {count:,}")

# 4. Define character set for CTC Loss
CTC_CHARACTERS = ['<BLANK>'] + [LABEL_TO_CHAR[int(label)] for label in unique_labels]
NUM_CLASSES = len(CTC_CHARACTERS)

print("-" * 60)
print(f"Total Classes for CRNN Model (including 1 <BLANK> token): {NUM_CLASSES}")

Unique label indices found in Train CSV : [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25]
Total unique label count                 : 26
Min label index: 0 | Max label index: 25
=== DIRECT INSPECTION RESULTS FROM TRAIN CSV ===
CSV Label:  0  -->  Letter: 'A'  |  Sample Count: 11,019
CSV Label:  1  -->  Letter: 'B'  |  Sample Count: 6,928
CSV Label:  2  -->  Letter: 'C'  |  Sample Count: 18,704
CSV Label:  3  -->  Letter: 'D'  |  Sample Count: 8,080
CSV Label:  4  -->  Letter: 'E'  |  Sample Count: 9,143
CSV Label:  5  -->  Letter: 'F'  |  Sample Count: 922
CSV Label:  6  -->  Letter: 'G'  |  Sample Count: 4,632
CSV Label:  7  -->  Letter: 'H'  |  Sample Count: 5,769
CSV Label:  8  -->  Letter: 'I'  |  Sample Count: 907
CSV Label:  9  -->  Letter: 'J'  |  Sample Count: 6,791
CSV Label: 10  -->  Letter: 'K'  |  Sample Count: 4,461
CSV Label: 11  -->  Letter: 'L'  |  Sample Count: 9,294
CSV Label: 12  -->  Letter: 'M'  |  Sample Count: 9,849
CSV Label: 13  

In [8]:
#-- Step 2.b: Split X,y and Train/Validation. --#
from sklearn.model_selection import train_test_split
#-- Separate Features (X) and Labels (y). --#
X_train_full = train_raw_df.iloc[:, 1:].values  # Pixel columns (from index 1 onwards)
y_train_full = train_raw_df.iloc[:, 0].values   # Label column (index 0)
#-- Split 80% for Training and 20% for Validation. --#
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, 
    y_train_full, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_train_full
)
print(f"Train samples      : {len(X_train):,}")
print(f"Validation samples : {len(X_val):,}")
print(f"Pixels per image   : {X_train.shape[1]}")


Train samples      : 238,368
Validation samples : 59,592
Pixels per image   : 784


In [9]:
#-- Step 2.c: Reshape, Normalize, and Create PyTorch DataLoaders. --#
#-- Import libraries. --#
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

In [10]:
#-- Automatically calculate image side length (sqrt(784) = 28). --#
side_len = int(np.sqrt(X_train.shape[1]))
#-- Reshape 1D -> 4D (Samples, Channel=1, 28, 28) & Normalize pixels to [0.0, 1.0]. --#
X_train_4d = X_train.reshape(-1, 1, side_len, side_len).astype('float32') / 255.0
X_val_4d = X_val.reshape(-1, 1, side_len, side_len).astype('float32') / 255.0
#-- onvert to PyTorch Tensors. --#
train_images_tensor = torch.tensor(X_train_4d, dtype=torch.float32)
train_labels_tensor = torch.tensor(y_train + 1, dtype=torch.long)
val_images_tensor = torch.tensor(X_val_4d, dtype=torch.float32)
val_labels_tensor = torch.tensor(y_val + 1, dtype=torch.long)
#-- Wrap tensors into PyTorch TensorDataset. --#
train_dataset = TensorDataset(train_images_tensor, train_labels_tensor)
val_dataset = TensorDataset(val_images_tensor, val_labels_tensor)
#-- Initialize DataLoaders. --#
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
# --- VERIFY TENSORS & DATALOADERS ---
print("--- DATALOADER SUMMARY ---")
print(f"Train Image Tensor Shape: {train_images_tensor.shape}")
print(f"Val Image Tensor Shape: {val_images_tensor.shape}")
print(f"Train Batches: {len(train_loader)}")
print(f"Validation Batches: {len(val_loader)}")

--- DATALOADER SUMMARY ---
Train Image Tensor Shape: torch.Size([238368, 1, 28, 28])
Val Image Tensor Shape: torch.Size([59592, 1, 28, 28])
Train Batches: 3725
Validation Batches: 932


# Step 3: Build CRNN Structure. #

In [11]:
#-- Part 3.1: Build CNN Feature Extractor. --#
#-- Step 3.a: Part 1 - CNN Feature Extractor Layer. --#
import torch
import torch.nn as nn
#-- Automatically select GPU or CPU device. --#
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Initialize CNN directly using nn.Sequential
cnn = nn.Sequential(
    #-- Block 1: (Batch, 1, 28, 28) -> (Batch, 32, 14, 14). --#
    nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
    nn.BatchNorm2d(32),
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=2, stride=2),
    #-- Block 2: (Batch, 32, 14, 14) -> (Batch, 64, 7, 7). --3
    nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
    nn.BatchNorm2d(64),
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=2, stride=2),
    #-- Block 3: Reduce Height (7 -> 1), keep Width = 7 -> (Batch, 128, 1, 7). --#
    nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
    nn.BatchNorm2d(128),
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=(7, 1))
).to(device)

print("Step 3.1: CNN Initialized......")
print(f"Device running: {device}")

Step 3.1: CNN Initialized......
Device running: cpu


In [12]:
#-- Part 3.2: BiLSTM, Linear Layer and Collect Parameters. --#
#-- BiLSTM Layer. --#
rnn = nn.LSTM(
    input_size=128, 
    hidden_size=128, 
    num_layers=2, 
    bidirectional=True, 
    batch_first=True
).to(device)
#-- Linear Classification Layer. --#
fc = nn.Linear(128 * 2, 27).to(device)
#-- Combine weights from all 3 layers into a list for the Optimizer. --#
model_params = list(cnn.parameters()) + list(rnn.parameters()) + list(fc.parameters())
print("--- Part 3.2: BiLSTM & FC Initialized. ---")
print(f"Total Trainable Parameters: {sum(p.numel() for p in model_params):,}")


--- Part 3.2: BiLSTM & FC Initialized. ---
Total Trainable Parameters: 759,515


In [13]:
#-- Step 3.c: Part 3 - Run Full Forward Pass Test on Data Batch. --#
#-- Fetch 1 real data batch from train_loader created in Step 2. --#
sample_images, sample_labels = next(iter(train_loader))
sample_images = sample_images.to(device)
#-- Pass through CNN. --#
cnn_out = cnn(sample_images)
#-- 2D Feature Map to 1D Sequence (squeeze Height=1, permute dimensions for BiLSTM). --#
sequence_in = cnn_out.squeeze(2).permute(0, 2, 1)
#-- Pass through BiLSTM. --#
rnn_out, _ = rnn(sequence_in)
#-- Pass through Linear layer to get Logits. --#
logits = fc(rnn_out)
# --- VERIFY FORWARD PASS RESULTS ---
print("--- Part 3.3: FORWARD PASS TEST. ---")
print(f"Input Images Batch Shape: {sample_images.shape}")
print(f"1. CNN Output Shape: {cnn_out.shape}")
print(f"2. Sequence Input Shape: {sequence_in.shape}")
print(f"3. BiLSTM Output Shape: {rnn_out.shape}")
print(f"4. Final Logits Shape: {logits.shape}  --> (Batch=64, Time_Steps=7, Classes=27)")

--- Part 3.3: FORWARD PASS TEST. ---
Input Images Batch Shape: torch.Size([64, 1, 28, 28])
1. CNN Output Shape: torch.Size([64, 128, 1, 7])
2. Sequence Input Shape: torch.Size([64, 7, 128])
3. BiLSTM Output Shape: torch.Size([64, 7, 256])
4. Final Logits Shape: torch.Size([64, 7, 27])  --> (Batch=64, Time_Steps=7, Classes=27)


In [14]:
#-- Step 4.a: Define Loss Function (CTC Loss) and Optimizer. --#
import torch.nn as nn
import torch.optim as optim

# 1. CTC Loss setup (Index 0 reserved for <BLANK>)
criterion = nn.CTCLoss(blank=0, zero_infinity=True)

# 2. Adam Optimizer (using model_params collected in Step 3.b)
optimizer = optim.Adam(model_params, lr=0.001)

print("=== STEP 4.a COMPLETE: CTC Loss & Optimizer Ready ===")
print(f"Optimizer : Adam (lr=0.001)")
print(f"Loss Func : CTCLoss (blank=0)")

=== STEP 4.a COMPLETE: CTC Loss & Optimizer Ready ===
Optimizer : Adam (lr=0.001)
Loss Func : CTCLoss (blank=0)


In [15]:
#-- Combine into one #
import torch
import torch.nn as nn
import torch.optim as optim
#-- Automatically select GPU or CPU device. --#
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#-- BLOCK 1: CNN Feature Extractor. ---#
cnn = nn.Sequential(
    #-- (Batch, 1, 28, 28) -> (Batch, 32, 14, 14). --#
    nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
    nn.BatchNorm2d(32),
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=2, stride=2),
    #-- (Batch, 32, 14, 14) -> (Batch, 64, 7, 7). --#
    nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
    nn.BatchNorm2d(64),
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=2, stride=2),
    #-- Reduce Height (7 -> 1), keep Width = 7 -> (Batch, 128, 1, 7). --#
    nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
    nn.BatchNorm2d(128),
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=(7, 1))
).to(device)
#-- BiLSTM Sequence Modeling. --#
rnn = nn.LSTM(
    input_size=128, 
    hidden_size=128, 
    num_layers=2, 
    bidirectional=True, 
    batch_first=True
).to(device)
#-- Linear Classifier (256 -> 27 classes). --#
fc = nn.Linear(128 * 2, 27).to(device)
#-- COLLECT PARAMETERS, LOSS FUNCTION & OPTIMIZER. ---#
model_params = list(cnn.parameters()) + list(rnn.parameters()) + list(fc.parameters())
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = optim.Adam(model_params, lr=0.001)
# 6. RUN TEST FORWARD PASS (Test 1 real batch from train_loader)
sample_images, sample_labels = next(iter(train_loader))
sample_images = sample_images.to(device)
# Sequential Forward Pass Pipeline
cnn_out = cnn(sample_images)
sequence_in = cnn_out.squeeze(2).permute(0, 2, 1)
rnn_out, _ = rnn(sequence_in)
logits = fc(rnn_out)
# --- PRINT CONSOLIDATED SUMMARY ---
print("=== FULL CRNN MODEL CONSOLIDATED SUCCESSFULLY ===")
print(f"Device Running: {device}")
print(f"Total Model Parameters: {sum(p.numel() for p in model_params):,}")
print(f"Input Images Shape: {sample_images.shape}")
print(f"Final Logits Shape: {logits.shape}  --> (Batch=64, Time_Steps=7, Classes=27)")
print("-" * 38)
print("CRNN Model is 100% ready for Training!")

=== FULL CRNN MODEL CONSOLIDATED SUCCESSFULLY ===
Device Running: cpu
Total Model Parameters: 759,515
Input Images Shape: torch.Size([64, 1, 28, 28])
Final Logits Shape: torch.Size([64, 7, 27])  --> (Batch=64, Time_Steps=7, Classes=27)
--------------------------------------
CRNN Model is 100% ready for Training!


# Step 4: Training Model. #

In [15]:
# ==========================================================
# STEP 4: TRAINING CRNN MODEL (LIMITED TO 5 EPOCHS)
# ==========================================================
import time
import torch

# 1. Giới hạn 5 Epochs & Cấu hình Early Stopping
MAX_EPOCHS = 5
PATIENCE = 3
patience_counter = 0
best_val_loss = float('inf')

total_steps = len(train_loader)

print("=== START TRAINING CRNN MODEL (MAX 5 EPOCHS) ===")

for epoch in range(MAX_EPOCHS):
    start_time = time.time()
    
    # ------------------------------------------------------
    # 1. TRAINING PHASE
    # ------------------------------------------------------
    cnn.train()
    rnn.train()
    fc.train()
    
    running_train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        batch_size = images.size(0)
        
        # Reset gradients
        optimizer.zero_grad()
        
        # Forward pass
        cnn_out = cnn(images)
        sequence_in = cnn_out.squeeze(2).permute(0, 2, 1)
        rnn_out, _ = rnn(sequence_in)
        logits = fc(rnn_out)
        
        # CTC Loss
        log_probs = logits.permute(1, 0, 2).log_softmax(2)
        input_lengths = torch.full(size=(batch_size,), fill_value=7, dtype=torch.long)
        target_lengths = torch.full(size=(batch_size,), fill_value=1, dtype=torch.long)
        
        loss = criterion(log_probs, labels, input_lengths, target_lengths)
        loss.backward()
        optimizer.step()
        
        running_train_loss += loss.item()
        
        # Train Accuracy (CTC Greedy Decoding)
        preds = logits.argmax(dim=2)
        for i in range(batch_size):
            pred_seq = preds[i].tolist()
            decoded_seq = []
            prev_token = None
            for token in pred_seq:
                if token != prev_token:
                    if token != 0:
                        decoded_seq.append(token)
                    prev_token = token
            if len(decoded_seq) > 0 and decoded_seq[0] == labels[i].item():
                train_correct += 1
            train_total += 1

    avg_train_loss = running_train_loss / len(train_loader)
    train_acc = train_correct / train_total

    # ------------------------------------------------------
    # 2. VALIDATION PHASE
    # ------------------------------------------------------
    cnn.eval()
    rnn.eval()
    fc.eval()
    
    running_val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            batch_size = images.size(0)
            
            cnn_out = cnn(images)
            sequence_in = cnn_out.squeeze(2).permute(0, 2, 1)
            rnn_out, _ = rnn(sequence_in)
            logits = fc(rnn_out)
            
            log_probs = logits.permute(1, 0, 2).log_softmax(2)
            input_lengths = torch.full(size=(batch_size,), fill_value=7, dtype=torch.long)
            target_lengths = torch.full(size=(batch_size,), fill_value=1, dtype=torch.long)
            
            val_loss = criterion(log_probs, labels, input_lengths, target_lengths)
            running_val_loss += val_loss.item()
            
            preds = logits.argmax(dim=2)
            for i in range(batch_size):
                pred_seq = preds[i].tolist()
                decoded_seq = []
                prev_token = None
                for token in pred_seq:
                    if token != prev_token:
                        if token != 0:
                            decoded_seq.append(token)
                        prev_token = token
                if len(decoded_seq) > 0 and decoded_seq[0] == labels[i].item():
                    val_correct += 1
                val_total += 1

    avg_val_loss = running_val_loss / len(val_loader)
    val_acc = val_correct / val_total
    
    # ------------------------------------------------------
    # 3. PRINT KERAS-STYLE PROGRESS LOG
    # ------------------------------------------------------
    elapsed_time = time.time() - start_time
    ms_per_step = int((elapsed_time / total_steps) * 1000)
    bar = "━" * 20
    
    print(f"Epoch {epoch+1}/{MAX_EPOCHS}")
    print(f"{total_steps}/{total_steps} {bar} {int(elapsed_time)}s {ms_per_step}ms/step - accuracy: {train_acc:.4f} - loss: {avg_train_loss:.4f} - val_accuracy: {val_acc:.4f} - val_loss: {avg_val_loss:.4f}")

    # ------------------------------------------------------
    # 4. SAVE BEST MODEL CHECKPOINT
    # ------------------------------------------------------
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        
        # Save Model Checkpoint
        torch.save({
            'cnn': cnn.state_dict(),
            'rnn': rnn.state_dict(),
            'fc': fc.state_dict()
        }, 'best_crnn_model.pt')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n[EARLY STOPPING TRIGGERED] Validation loss did not improve for {PATIENCE} consecutive epochs.")
            break

# ------------------------------------------------------
# 5. RESTORE BEST WEIGHTS
# ------------------------------------------------------
print("\n=== RESTORING BEST MODEL WEIGHTS ===")
checkpoint = torch.load('best_crnn_model.pt')
cnn.load_state_dict(checkpoint['cnn'])
rnn.load_state_dict(checkpoint['rnn'])
fc.load_state_dict(checkpoint['fc'])
print("Successfully restored best model weights from 'best_crnn_model.pt'!")

=== START TRAINING CRNN MODEL (MAX 5 EPOCHS) ===
Epoch 1/5
3725/3725 ━━━━━━━━━━━━━━━━━━━━ 308s 82ms/step - accuracy: 0.9888 - loss: 0.0393 - val_accuracy: 0.9865 - val_loss: 0.0465
Epoch 2/5
3725/3725 ━━━━━━━━━━━━━━━━━━━━ 409s 109ms/step - accuracy: 0.9903 - loss: 0.0331 - val_accuracy: 0.9909 - val_loss: 0.0331
Epoch 3/5
3725/3725 ━━━━━━━━━━━━━━━━━━━━ 402s 107ms/step - accuracy: 0.9919 - loss: 0.0272 - val_accuracy: 0.9891 - val_loss: 0.0375
Epoch 4/5
3725/3725 ━━━━━━━━━━━━━━━━━━━━ 405s 108ms/step - accuracy: 0.9929 - loss: 0.0234 - val_accuracy: 0.9882 - val_loss: 0.0409
Epoch 5/5
3725/3725 ━━━━━━━━━━━━━━━━━━━━ 406s 109ms/step - accuracy: 0.9937 - loss: 0.0203 - val_accuracy: 0.9915 - val_loss: 0.0315

=== RESTORING BEST MODEL WEIGHTS ===
Successfully restored best model weights from 'best_crnn_model.pt'!


# Step 5: Test Model. #

In [17]:
# ==========================================================
# CELL 1: INITIALIZE MODEL ARCHITECTURE & LOAD WEIGHTS
# ==========================================================
import os
import glob
import torch
import torch.nn as nn

# 1. Select device (GPU / CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"⚡ Running on device: {device}")

# 2. Re-create the exact CRNN architecture used during training
cnn = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
    nn.BatchNorm2d(32),
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=2, stride=2),
    
    nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
    nn.BatchNorm2d(64),
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=2, stride=2),
    
    nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
    nn.BatchNorm2d(128),
    nn.ReLU(inplace=True),
    nn.MaxPool2d(kernel_size=(7, 1))
).to(device)

rnn = nn.LSTM(
    input_size=128, 
    hidden_size=128, 
    num_layers=2, 
    bidirectional=True, 
    batch_first=True
).to(device)

fc = nn.Linear(128 * 2, 27).to(device)

# 3. Load saved weights from checkpoint
model_path = '../Models/best_crnn_model.pt'

if os.path.exists(model_path):
    print(f"📂 Found model checkpoint at: {model_path}")
    checkpoint = torch.load(model_path, map_location=device)
    cnn.load_state_dict(checkpoint['cnn'])
    rnn.load_state_dict(checkpoint['rnn'])
    fc.load_state_dict(checkpoint['fc'])
    print("✅ Successfully restored model weights!")
else:
    print(f"⚠️ Warning: Checkpoint path '{model_path}' does not exist!")

# 4. Set model to evaluation mode
cnn.eval()
rnn.eval()
fc.eval()

# 5. Scan and prepare test files
test_dir = '../Datasets/Test/EngText'
test_files = glob.glob(os.path.join(test_dir, "*.csv")) + glob.glob(os.path.join(test_dir, "*.icsv"))
test_files = sorted(list(set(test_files)))

print("\n" + "=" * 65)
print(f"🔍 READY TO TEST {len(test_files)} FILE(S) IN '{test_dir}':")
for file in test_files:
    print(f"  • {os.path.basename(file)}")
print("=" * 65)

⚡ Running on device: cpu
📂 Found model checkpoint at: ../Models/best_crnn_model.pt
✅ Successfully restored model weights!

🔍 READY TO TEST 4 FILE(S) IN '../Datasets/Test/EngText':
  • Datasets_01.csv
  • Datasets_02.csv
  • Datasets_03.csv
  • Datasets_04.csv


In [18]:
# ==========================================================
# CELL 2: EVALUATE MODEL ON TEST FILES
# ==========================================================
import pandas as pd
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

# Dictionary to map Label Index (0–25) -> Letter ('A'–'Z')
LABEL_TO_CHAR = {i: chr(65 + i) for i in range(26)}

total_overall_correct = 0
total_overall_samples = 0

print("🚀 STARTING EVALUATION...\n")

for file_path in test_files:
    file_name = os.path.basename(file_path)
    print(f"📄 Testing file: {file_name}")
    
    # Read dataset
    df = pd.read_csv(file_path)
    
    # Column 0: Label, Columns 1 to 784: Pixels
    y_test_raw = df.iloc[:, 0].values               
    X_test_raw = df.iloc[:, 1:].values              
    
    # Reshape pixel array into (N, 1, 28, 28) and normalize to [0.0, 1.0]
    side_len = int(np.sqrt(X_test_raw.shape[1]))
    X_test_4d = X_test_raw.reshape(-1, 1, side_len, side_len).astype('float32') / 255.0
    
    # Target label offset by +1 (index 0 reserved for CTC <BLANK>)
    y_test_target = y_test_raw + 1
    
    test_tensor_x = torch.tensor(X_test_4d, dtype=torch.float32)
    test_tensor_y = torch.tensor(y_test_target, dtype=torch.long)
    
    test_dataset = TensorDataset(test_tensor_x, test_tensor_y)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    file_correct = 0
    file_total = 0
    sample_results = []
    
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device)
            batch_size = batch_x.size(0)
            
            # Forward Pass
            cnn_out = cnn(batch_x)
            sequence_in = cnn_out.squeeze(2).permute(0, 2, 1)
            rnn_out, _ = rnn(sequence_in)
            logits = fc(rnn_out)
            
            # CTC Greedy Decoding
            preds = logits.argmax(dim=2)
            
            for i in range(batch_size):
                pred_seq = preds[i].tolist()
                decoded_seq = []
                prev_token = None
                
                # Remove repeated tokens and <BLANK> (0)
                for token in pred_seq:
                    if token != prev_token:
                        if token != 0:
                            decoded_seq.append(token)
                        prev_token = token
                
                pred_token = decoded_seq[0] if len(decoded_seq) > 0 else -1
                gt_token = batch_y[i].item()
                
                is_correct = (pred_token == gt_token)
                if is_correct:
                    file_correct += 1
                file_total += 1
                
                # Store first 10 results for display
                if len(sample_results) < 10:
                    gt_char = LABEL_TO_CHAR.get(gt_token - 1, 'Unknown')
                    pred_char = LABEL_TO_CHAR.get(pred_token - 1, 'None') if pred_token != -1 else 'None'
                    sample_results.append({
                        'GT Label': gt_token - 1,
                        'GT Char': gt_char,
                        'Pred Label': pred_token - 1 if pred_token != -1 else -1,
                        'Pred Char': pred_char,
                        'Result': '✅ Correct' if is_correct else '❌ Wrong'
                    })
    
    acc = (file_correct / file_total) * 100 if file_total > 0 else 0
    print(f"🎯 Accuracy [{file_name}]: {acc:.2f}% ({file_correct}/{file_total} correct)")
    
    # Display sample DataFrame
    sample_df = pd.DataFrame(sample_results)
    display(sample_df)
    
    total_overall_correct += file_correct
    total_overall_samples += file_total

# Summary
print("\n" + "=" * 65)
if total_overall_samples > 0:
    overall_acc = (total_overall_correct / total_overall_samples) * 100
    print("🏆 OVERALL SUMMARY:")
    print(f"📊 Overall Accuracy: {overall_acc:.2f}% ({total_overall_correct}/{total_overall_samples} samples correct)")
print("=" * 65)

🚀 STARTING EVALUATION...

📄 Testing file: Datasets_01.csv
🎯 Accuracy [Datasets_01.csv]: 99.19% (19838/20000 correct)


,GT Label,GT Char,Pred Label,Pred Char,Result
0,20,U,20,U,✅ Correct
1,14,O,14,O,✅ Correct
2,4,E,4,E,✅ Correct
3,20,U,20,U,✅ Correct
4,14,O,14,O,✅ Correct
5,14,O,14,O,✅ Correct
6,15,P,15,P,✅ Correct
7,0,A,0,A,✅ Correct
8,16,Q,16,Q,✅ Correct
9,24,Y,24,Y,✅ Correct


📄 Testing file: Datasets_02.csv
🎯 Accuracy [Datasets_02.csv]: 99.24% (19848/20000 correct)


,GT Label,GT Char,Pred Label,Pred Char,Result
0,13,N,13,N,✅ Correct
1,14,O,14,O,✅ Correct
2,4,E,4,E,✅ Correct
3,2,C,2,C,✅ Correct
4,18,S,18,S,✅ Correct
5,22,W,22,W,✅ Correct
6,14,O,14,O,✅ Correct
7,18,S,18,S,✅ Correct
8,15,P,15,P,✅ Correct
9,2,C,2,C,✅ Correct


📄 Testing file: Datasets_03.csv
🎯 Accuracy [Datasets_03.csv]: 99.06% (19812/20000 correct)


,GT Label,GT Char,Pred Label,Pred Char,Result
0,20,U,20,U,✅ Correct
1,20,U,20,U,✅ Correct
2,19,T,19,T,✅ Correct
3,0,A,0,A,✅ Correct
4,12,M,12,M,✅ Correct
5,1,B,1,B,✅ Correct
6,24,Y,24,Y,✅ Correct
7,3,D,3,D,✅ Correct
8,15,P,15,P,✅ Correct
9,14,O,14,O,✅ Correct


📄 Testing file: Datasets_04.csv
🎯 Accuracy [Datasets_04.csv]: 99.05% (14352/14490 correct)


,GT Label,GT Char,Pred Label,Pred Char,Result
0,24,Y,24,Y,✅ Correct
1,11,L,11,L,✅ Correct
2,13,N,13,N,✅ Correct
3,10,K,10,K,✅ Correct
4,20,U,20,U,✅ Correct
5,18,S,18,S,✅ Correct
6,18,S,18,S,✅ Correct
7,1,B,1,B,✅ Correct
8,18,S,18,S,✅ Correct
9,12,M,12,M,✅ Correct



🏆 OVERALL SUMMARY:
📊 Overall Accuracy: 99.14% (73850/74490 samples correct)
